# Lecture 8: Feature Engineering
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Apply log, sqrt, and Box-Cox transformations to skewed features
2. Create derived features from domain knowledge
3. Encode categorical variables correctly
4. Apply and compare feature scaling methods
5. Perform feature selection using filter and embedded methods

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                    RobustScaler, LabelEncoder)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print('Ready.')

## 8.1 Simulating a Kenya Household Survey Dataset

In [ ]:
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'county':            np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'], n),
    'household_size':    np.random.randint(1, 10, n),
    'monthly_income':    np.random.exponential(scale=25000, size=n).clip(2000, 300000),
    'education_level':   np.random.choice(['None','Primary','Secondary','University'], n,
                                          p=[0.1, 0.3, 0.4, 0.2]),
    'employment_status': np.random.choice(['Employed','Self-employed','Unemployed'], n,
                                          p=[0.45, 0.35, 0.20]),
    'expenditure_food':  None,
    'expenditure_health':None,
    'years_in_county':   np.random.randint(0, 40, n),
})
df['expenditure_food']   = (df['monthly_income'] * np.random.uniform(0.3, 0.6, n)).round(0)
df['expenditure_health'] = (df['monthly_income'] * np.random.uniform(0.02, 0.15, n)).round(0)
df['poverty_status'] = ((df['monthly_income'] / df['household_size']) < 3000).astype(int)

print(f'Shape: {df.shape}')
print(f'Poverty rate: {df.poverty_status.mean()*100:.1f}%')
df.head()

## 8.2 Handling Skewed Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (title, data) in enumerate([
    ('Original (income)', df['monthly_income']),
    ('Log Transform',     np.log1p(df['monthly_income'])),
    ('Sqrt Transform',    np.sqrt(df['monthly_income'])),
]):
    axes[0, i].hist(data, bins=40, color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'{title}\nSkew: {pd.Series(data).skew():.2f}')
    stats.probplot(data, plot=axes[1, i])
    axes[1, i].set_title(f'Q-Q Plot: {title}')
plt.tight_layout()
plt.show()

In [ ]:
df['income_log']  = np.log1p(df['monthly_income'])
df['income_sqrt'] = np.sqrt(df['monthly_income'])
print('Log transform skew:', df.income_log.skew().round(3))
print('Sqrt transform skew:', df.income_sqrt.skew().round(3))

## 8.3 Creating Derived Features

In [ ]:
# Domain-knowledge derived features
df['food_expenditure_ratio']   = df['expenditure_food']   / df['monthly_income']
df['health_expenditure_ratio'] = df['expenditure_health'] / df['monthly_income']
df['income_per_person']        = df['monthly_income'] / df['household_size']

# Correlation with poverty status
derived = ['food_expenditure_ratio', 'health_expenditure_ratio',
           'income_per_person', 'monthly_income']
print('Correlation of features with poverty_status:')
for col in derived:
    corr = df[col].corr(df['poverty_status'])
    print(f'  {col:30s}: {corr:.4f}')

## 8.4 Encoding Categorical Variables

In [ ]:
# Ordinal encoding for education
educ_map = {'None': 0, 'Primary': 1, 'Secondary': 2, 'University': 3}
df['education_encoded'] = df['education_level'].map(educ_map)

# One-hot encoding for county and employment_status
df_encoded = pd.get_dummies(df, columns=['county', 'employment_status'], drop_first=False)

print(f'Shape after encoding: {df_encoded.shape}')
new_cols = [c for c in df_encoded.columns if 'county_' in c or 'employment_' in c]
print(f'New columns created: {new_cols}')

## 8.5 Feature Scaling Comparison

In [ ]:
num_feats = ['monthly_income', 'household_size', 'expenditure_food', 'years_in_county']
X_num = df[num_feats]

scalers = {'MinMax': MinMaxScaler(), 'Standard': StandardScaler(), 'Robust': RobustScaler()}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, sc) in zip(axes, scalers.items()):
    scaled = pd.DataFrame(sc.fit_transform(X_num), columns=num_feats)
    scaled.boxplot(ax=ax)
    ax.set_title(f'{name} Scaling')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8.6 Feature Selection

In [ ]:
feature_cols = ['household_size', 'income_log', 'food_expenditure_ratio',
                'health_expenditure_ratio', 'income_per_person', 'education_encoded',
                'years_in_county'] + \
               [c for c in df_encoded.columns if 'county_' in c or 'employment_' in c]

X_final = df_encoded[feature_cols].fillna(0)
y_final = df_encoded['poverty_status']

# Filter method: ANOVA F-test
selector = SelectKBest(f_classif, k=8)
selector.fit(X_final, y_final)
scores = pd.DataFrame({'feature': feature_cols,
                       'f_score': selector.scores_}).sort_values('f_score', ascending=False)
print('Top 8 features by ANOVA F-score:')
print(scores.head(8).to_string(index=False))

In [ ]:
# Embedded method: Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_final, y_final)
fi = pd.DataFrame({'feature': feature_cols,
                   'importance': rf.feature_importances_})\
       .sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
plt.barh(fi.feature[::-1], fi.importance[::-1], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Top 10 Features — Random Forest Embedded Selection')
plt.tight_layout()
plt.show()

### Exercise

Using the household survey DataFrame:

1. Create a new feature `expenditure_ratio` = (expenditure_food + expenditure_health) / monthly_income.
2. Calculate its correlation with poverty_status. Is it more or less predictive than food_expenditure_ratio alone?
3. Apply RFE (Recursive Feature Elimination) with a LogisticRegression estimator to select the top 5 features.
4. Train a RandomForestClassifier using only the top 5 RFE features and compare test accuracy to the full feature set.

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*